In [ ]:
import os
import glob
import time
import pickle
from collections import deque, defaultdict
from contextlib import contextmanager

import cv2
import numpy as np
import mediapipe as mp
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Masking

from feedback_analyzer import compute_feedback
from dtaidistance import dtw_ndim

# ==========================================
# 0. FEATURE ENGINEERING  (67-dim per-frame vector)
# ==========================================
LANDMARK_INDICES = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]

LS, RS = 0, 1
LE, RE = 2, 3
LW, RW = 4, 5
LHIP, RHIP = 6, 7
LKNEE, RKNEE = 8, 9


def get_hip_center(frame):
    return np.array([(frame[LHIP * 2] + frame[RHIP * 2]) / 2.0,
                     (frame[LHIP * 2 + 1] + frame[RHIP * 2 + 1]) / 2.0])


def get_shoulder_center(frame):
    return np.array([(frame[LS * 2] + frame[RS * 2]) / 2.0,
                     (frame[LS * 2 + 1] + frame[RS * 2 + 1]) / 2.0])


def translation_normalization(frame):
    hip_center = get_hip_center(frame)
    normalized = []
    for i in range(0, len(frame), 2):
        joint_xy = np.array([frame[i], frame[i + 1]]) - hip_center
        normalized.extend(joint_xy.tolist())
    return normalized


def scale_normalization(frame):
    torso = np.linalg.norm(get_shoulder_center(frame) - get_hip_center(frame))
    if torso < 1e-6:
        torso = 1e-6
    return [v / torso for v in frame]


def calculate_angle(a, b, c):
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = abs((180.0 * radians) / np.pi)
    return 360.0 - angle if angle > 180.0 else angle


def extract_angles(frame):
    def get_xy(idx):
        return [frame[idx * 2], frame[idx * 2 + 1]]
    return [
        calculate_angle(get_xy(LS), get_xy(LE), get_xy(LW)),
        calculate_angle(get_xy(RS), get_xy(RE), get_xy(RW)),
        calculate_angle(get_xy(LHIP), get_xy(LS), get_xy(LE)),
        calculate_angle(get_xy(RHIP), get_xy(RS), get_xy(RE)),
        calculate_angle(get_xy(LS), get_xy(LHIP), get_xy(LKNEE)),
        calculate_angle(get_xy(RS), get_xy(RHIP), get_xy(RKNEE)),
    ]


def extract_velocities(curr_frame, prev_frame):
    velocities = []
    for j_id in [LS, RS, LE, RE, LW, RW]:
        velocities.extend([curr_frame[j_id * 2] - prev_frame[j_id * 2],
                           curr_frame[j_id * 2 + 1] - prev_frame[j_id * 2 + 1]])
    return velocities


def extract_wrist_speeds(velocities):
    return [np.sqrt(velocities[8] ** 2 + velocities[9] ** 2),
            np.sqrt(velocities[10] ** 2 + velocities[11] ** 2)]


def wrist_to_shoulder_distance(frame):
    def get_xy(idx):
        return np.array([frame[idx * 2], frame[idx * 2 + 1]])
    return [np.linalg.norm(get_xy(LW) - get_xy(LS)), np.linalg.norm(get_xy(RW) - get_xy(RS))]


def lead_rear_asymmetry(frame):
    return [frame[LW * 2] - frame[RW * 2]]


def wrist_motion_angle(velocities):
    return [np.arctan2(velocities[9], velocities[8]), np.arctan2(velocities[11], velocities[10])]


def cumulative_wrist_displacement(curr_frame, frame0):
    def get_xy(frame, idx):
        return frame[idx * 2], frame[idx * 2 + 1]
    lw_x0, lw_y0 = get_xy(frame0, LW)
    rw_x0, rw_y0 = get_xy(frame0, RW)
    lw_x, lw_y = get_xy(curr_frame, LW)
    rw_x, rw_y = get_xy(curr_frame, RW)
    return [lw_y - lw_y0, rw_y - rw_y0], [lw_x - lw_x0, rw_x - rw_x0]


def vertical_horizontal_ratio(vert_disp, horiz_disp):
    return [abs(vert_disp[0]) / (abs(horiz_disp[0]) + 1e-6),
            abs(vert_disp[1]) / (abs(horiz_disp[1]) + 1e-6)]


def elbow_shoulder_height(frame):
    return [frame[LE * 2 + 1] - frame[LS * 2 + 1], frame[RE * 2 + 1] - frame[RS * 2 + 1]]


def elbow_torso_offset(frame):
    hip_center_x = (frame[LHIP * 2] + frame[RHIP * 2]) / 2.0
    return [frame[LE * 2] - hip_center_x, frame[RE * 2] - hip_center_x]


def build_feature_vector(curr_frame, prev_frame, frame0):
    angles = extract_angles(curr_frame)
    velocities = extract_velocities(curr_frame, prev_frame)
    speeds = extract_wrist_speeds(velocities)
    wrist_dist = wrist_to_shoulder_distance(curr_frame)
    asymmetry = lead_rear_asymmetry(curr_frame)
    motion_angle = wrist_motion_angle(velocities)
    vert_disp, horiz_disp = cumulative_wrist_displacement(curr_frame, frame0)
    vh_ratio = vertical_horizontal_ratio(vert_disp, horiz_disp)
    return np.concatenate([
        curr_frame, velocities, angles, speeds, wrist_dist, asymmetry,
        motion_angle, vert_disp, horiz_disp, vh_ratio,
        elbow_shoulder_height(curr_frame), elbow_torso_offset(curr_frame)
    ])


def process_punch_sequence(punch):
    """punch: list of 32-dim normalized frames for ONE isolated clip.
    Returns list of 67-dim feature vectors, frame0-anchored (matches training)."""
    if len(punch) == 0:
        return []
    zero_vel, zero_speed, zero_asym = [0.0] * 12, [0.0, 0.0], [0.0]
    frame0 = punch[0]
    v0, h0 = cumulative_wrist_displacement(punch[0], frame0)
    features = [np.concatenate([
        punch[0], zero_vel, extract_angles(punch[0]), zero_speed,
        wrist_to_shoulder_distance(punch[0]), zero_asym,
        wrist_motion_angle(zero_vel), v0, h0, vertical_horizontal_ratio(v0, h0),
        elbow_shoulder_height(punch[0]), elbow_torso_offset(punch[0])
    ])]
    for i in range(1, len(punch)):
        features.append(build_feature_vector(punch[i], punch[i - 1], frame0))
    return features


# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
INPUT_VIDEO = r"C:\Users\abdul\OneDrive\Desktop\AI BOXING ASSISTANT\TestVideos\NO_HIP_SHOULDER_MOVEMENT.mp4"
OUTPUT_DIR = r"C:\Users\abdul\OneDrive\Desktop\AI BOXING ASSISTANT\Test Clips"

# Reference punches: REFERENCE_ROOT\<punch folder>\*.pkl  (each .pkl holds 5 sequences)
# Feedback is computed against EVERY reference sequence of the predicted punch type and
# a message is only shown if a strict majority of references agree on it.
REFERENCE_ROOT = r"C:\Users\abdul\OneDrive\Desktop\AI BOXING ASSISTANT\goldenStandardPunches"

MODEL1_PATH = r"C:\Users\abdul\Downloads\model_fixed.h5"    # jab | leadhook | cross_rearhook | uppercuts_nopunch
MODEL2_PATH = r"C:\Users\abdul\Downloads\model2_fixed.h5"   # leaduppercut | rearuppercut | nopunch
MODEL3_PATH = r"C:\Users\abdul\Downloads\model3_fixed.h5"   # cross | rearhook

# Set False to skip reference lookup + DTW + feedback text (clips are still exported)
ENABLE_FEEDBACK = True

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Trimming & pipeline constants (unchanged)
VELOCITY_THRESHOLD = 0.008
PRE_PAD_LEAD, POST_PAD_LEAD = 12, 12
PRE_PAD_REAR, POST_PAD_REAR = 18, 20

MAX_FRAMES = 110            # time steps the models were trained with
MODEL_TARGET_FRAMES = MAX_FRAMES

LEAD_THRESHOLD = 0.15
REAR_THRESHOLD = 0.10
UPPERCUT_Y_THRESHOLD = 0.30

WINDOW_BEFORE, WINDOW_AFTER = 60, 60
TOTAL_WINDOW = WINDOW_BEFORE + WINDOW_AFTER + 1

# MediaPipe indexes removed after Z-removal (same as the training pipeline)
REMOVE_INDEXES = set([
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45
])

# ---- model class mappings (index order MUST match training) ----
M1_CLASSES = {0: "Jab", 1: "Lead Hook", 2: "Cross/Rear Hook", 3: "Uppercut/NoPunch"}
M2_CLASSES = {0: "Lead Uppercut", 1: "Rear Uppercut", 2: "NoPunch"}
M3_CLASSES = {0: "Cross", 1: "Rear Hook"}

# Per-frame feature size each model was trained with. Only used if load_model()
# fails and the architecture has to be rebuilt before loading the weights.
FEATURE_DIM_M1 = 67
FEATURE_DIM_M2 = 67
FEATURE_DIM_M3 = 67

# When Model 2 says NoPunch we revert to Model 1's answer. Model 1's merged
# class "Uppercut/NoPunch" can't be output as-is, so it becomes NoPunch.
M1_FALLBACK = {"Jab": "Jab", "Lead Hook": "Lead Hook", "Uppercut/NoPunch": "NoPunch"}
M1_TO_M3 = {"Cross/Rear Hook"}

LABEL_TO_FOLDER = {
    "Jab": "jab", "Cross": "cross", "Lead Hook": "leadhook", "Rear Hook": "rearhook",
    "Lead Uppercut": "leaduppercut", "Rear Uppercut": "rearuppercut",
}
LABEL_TO_PUNCH_TYPE = {
    "Jab": "jab", "Cross": "cross", "Lead Hook": "lead_hook", "Rear Hook": "rear_hook",
    "Lead Uppercut": "lead_uppercut", "Rear Uppercut": "rear_uppercut",
}

# Must match the string compute_feedback() appends when nothing fires
EXCELLENT_MSG = "Excellent form! Great execution."

_REFERENCE_CACHE = {}


# ==========================================
# 1a. MODEL LOADING
# ==========================================
def build_model(num_classes, feature_dim, max_frames=MAX_FRAMES):
    """Exact architecture used in training (same for all three models)."""
    return Sequential([
        Input(shape=(max_frames, feature_dim)),
        Masking(mask_value=0.0),
        LSTM(64, return_sequences=True),
        Dropout(0.4),
        LSTM(32, return_sequences=False),
        Dropout(0.4),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax'),
    ])


def load_punch_model(path, num_classes, feature_dim):
    """1) try a normal load_model (full-model .h5)
       2) otherwise rebuild the architecture and load the weights from the .h5"""
    try:
        m = tf.keras.models.load_model(path, compile=False)
        how = "load_model"
    except Exception as e:
        print(f"  load_model failed ({type(e).__name__}); rebuilding architecture + loading weights")
        m = build_model(num_classes, feature_dim)
        m.load_weights(path)
        how = "rebuilt architecture + load_weights"
    return m, how


# ==========================================
# 1b. TIMING
# ==========================================
class Timer:
    """Accumulates wall time per named, NON-overlapping stage."""
    def __init__(self):
        self.t = defaultdict(float)
        self.n = defaultdict(int)

    def add(self, name, seconds):
        self.t[name] += seconds
        self.n[name] += 1

    @contextmanager
    def section(self, name):
        s = time.perf_counter()
        try:
            yield
        finally:
            self.add(name, time.perf_counter() - s)


def print_timing_report(timer, wall_time, frames, punches, load_time, out_path):
    accounted = sum(timer.t.values())
    other = max(0.0, wall_time - accounted)
    rows = sorted(timer.t.items(), key=lambda kv: kv[1], reverse=True)
    rows.append(("unaccounted (loop overhead, etc.)", other))

    lines = []
    lines.append("=" * 86)
    lines.append(" TIMING REPORT")
    lines.append("=" * 86)
    lines.append(f" Model load + warm-up (not in wall time): {load_time:8.2f} s")
    lines.append(f" Frames processed: {frames}   Punches found: {punches}")
    lines.append(f" Total wall time : {wall_time:8.2f} s   ({frames / max(wall_time, 1e-9):.2f} frames/s)")
    lines.append("-" * 86)
    lines.append(f" {'stage':<36}{'total (s)':>11}{'% of wall':>11}{'calls':>9}{'avg (ms)':>12}")
    lines.append("-" * 86)
    for name, secs in rows:
        calls = timer.n.get(name, 0)
        avg_ms = (secs / calls * 1000) if calls else float("nan")
        calls_s = f"{calls}" if calls else "-"
        avg_s = f"{avg_ms:.2f}" if calls else "-"
        lines.append(f" {name:<36}{secs:>11.2f}{100 * secs / max(wall_time, 1e-9):>10.1f}%{calls_s:>9}{avg_s:>12}")
    lines.append("=" * 86)
    report = "\n".join(lines)
    print("\n" + report)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(report)


# ==========================================
# 2. HELPERS
# ==========================================
def get_wrist_distances(landmarks):
    nose = np.array([landmarks[0].x, landmarks[0].y])
    l_wrist = np.array([landmarks[15].x, landmarks[15].y])
    r_wrist = np.array([landmarks[16].x, landmarks[16].y])
    return np.linalg.norm(l_wrist - nose), np.linalg.norm(r_wrist - nose)


def compute_streaming_metrics(norm, prev_norm):
    """Frame-to-frame values used ONLY for live trigger / boundary detection."""
    if prev_norm is None:
        velocities = [0.0] * 12
    else:
        velocities = extract_velocities(norm, prev_norm)
    lw_speed, rw_speed = extract_wrist_speeds(velocities)
    angles = extract_angles(norm)
    return lw_speed, rw_speed, angles[0], angles[1]


def prepare_model_input(clip_features, model):
    """Take the leading F features (F = model's input size), then pad/trim to the
    model's time dimension. Returns shape (1, T, F) float32."""
    t_dim = model.input_shape[1] or MODEL_TARGET_FRAMES
    f_dim = model.input_shape[-1]
    seq = np.array(clip_features)[:, :f_dim]
    frames = seq.shape[0]
    if frames > t_dim:
        seq = seq[:t_dim]
    elif frames < t_dim:
        seq = np.vstack([seq, np.zeros((t_dim - frames, f_dim))])
    return np.expand_dims(seq, axis=0).astype(np.float32)


def run_model(model, x):
    """Direct call is faster than model.predict() for a single clip."""
    return np.asarray(model(x, training=False))[0]


# ==========================================
# 2b. REFERENCE PUNCHES FROM .pkl  (replaces the reference-video pipeline)
#
#   Each .pkl holds 5 sequences in the same raw format as the training dataset
#   (frames of 99 MediaPipe values: x,y,z for 33 landmarks). Every sequence goes
#   through: drop z -> drop noisy joints -> translation/scale normalisation ->
#   process_punch_sequence -> first 55 features. That is the same recipe the
#   user's clip goes through, so features line up for DTW + compute_feedback.
#   Raw ankle coordinates (before normalisation) are kept for the stance metric.
# ==========================================
def clean_raw_sequence(seq):
    """99 values/frame (x,y,z) or 66 (x,y) or already-cleaned 32 -> list of 32-value frames."""
    cleaned = []
    for frame in seq:
        vals = np.asarray(frame, dtype=np.float64).ravel().tolist()
        n = len(vals)
        if n == 99:
            vals = [v for i, v in enumerate(vals) if (i % 3) != 2]
            n = 66
        if n == 66:
            vals = [v for i, v in enumerate(vals) if i not in REMOVE_INDEXES]
        elif n != 32:
            raise ValueError(f"unexpected frame length {n} (expected 99, 66 or 32)")
        cleaned.append(vals)
    return cleaned


def load_reference_sequences(pkl_path):
    """Returns a list of raw sequences from one .pkl file."""
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    if isinstance(data, dict):
        data = list(data.values())
    if isinstance(data, np.ndarray):
        data = list(data) if data.ndim == 3 else [data]
    elif isinstance(data, (list, tuple)):
        # a list of frames (single sequence) vs. a list of sequences
        if len(data) > 0 and np.asarray(data[0], dtype=np.float64).ndim == 1:
            data = [data]
    else:
        raise ValueError(f"unsupported pickle content type: {type(data)}")
    return data


def build_reference_entry(raw_seq, source):
    cleaned = clean_raw_sequence(raw_seq)
    # raw (un-normalised) ankle coords: joints 10/11 = left/right ankle -> values 20..23
    ankle = [(f[20], f[21], f[22], f[23]) for f in cleaned]
    norm = [scale_normalization(translation_normalization(f)) for f in cleaned]
    feats = process_punch_sequence(norm)
    return {
        "seq": [np.asarray(v)[:55] for v in feats],
        "ankle": ankle,
        "source": source,
    }


def get_reference_data(predicted_label):
    """Returns a list of reference entries for this punch type, or None. Cached per label."""
    if predicted_label in _REFERENCE_CACHE:
        return _REFERENCE_CACHE[predicted_label]

    folder_name = LABEL_TO_FOLDER.get(predicted_label)
    refs = []
    if folder_name is not None:
        folder_path = os.path.join(REFERENCE_ROOT, folder_name)
        if not os.path.isdir(folder_path):
            print(f"[WARN] Reference folder not found: {folder_path}")
        else:
            pkl_files = sorted(glob.glob(os.path.join(folder_path, "**", "*.pkl"), recursive=True))
            if not pkl_files:
                print(f"[WARN] No .pkl reference files found in: {folder_path}")
            seqs = []
            for p in pkl_files:
                try:
                    seqs = load_reference_sequences(p)
                except Exception as e:
                    print(f"[WARN] Could not read {p}: {e}")
                    continue
                for k, s in enumerate(seqs):
                    if len(s) < 2:
                        continue
                    try:
                        refs.append(build_reference_entry(s, f"{os.path.basename(p)}#{k}"))
                    except Exception as e:
                        print(f"[WARN] Skipped {os.path.basename(p)}#{k}: {e}")
            if refs:
                n_files = len(pkl_files)
                first_len = np.asarray(seqs[0][0]).size if seqs and len(seqs[0]) else "?"
                print(f"[ref] {predicted_label}: {len(refs)} reference sequence(s) from {n_files} file(s) "
                      f"(raw frame length {first_len}, first sequence {len(refs[0]['seq'])} frames)")

    result = refs if refs else None
    _REFERENCE_CACHE[predicted_label] = result
    return result


# ==========================================
# 2c. FEEDBACK ENSEMBLE (majority vote across ALL references)

def run_feedback_ensemble(user_seq_55, refs, punch_type, clip_ankle, window=20):
    """Returns (final_feedback, votes, n_refs_used, per_ref_logs)."""
    u_arr = np.array(user_seq_55, dtype=np.double)
    votes = defaultdict(int)
    first_seen = {}
    per_ref_logs = []
    n_used = 0

    for r in refs:
        r_arr = np.array(r["seq"], dtype=np.double)
        try:
            path = dtw_ndim.warping_path(u_arr, r_arr, window=window)
        except Exception as e:
            print(f"[WARN] DTW failed against {r['source']}: {e}")
            continue

        log = []
        fb = compute_feedback(user_seq_55, r["seq"], path, punch_type,
                              clip_ankle, r["ankle"], log)
        n_used += 1
        per_ref_logs.append((r["source"], log))

        for msg in fb:
            if msg == EXCELLENT_MSG:
                continue
            votes[msg] += 1
            first_seen.setdefault(msg, len(first_seen))

    if n_used == 0:
        return ["Could not compare against any reference."], {}, 0, per_ref_logs

    needed = n_used // 2 + 1                       # strict majority (3 of 5)
    final = [m for m in sorted(votes, key=lambda m: first_seen[m]) if votes[m] >= needed]
    if not final:
        final = [EXCELLENT_MSG]
    return final, dict(votes), n_used, per_ref_logs


# ==========================================
# 3. ADAPTIVE MOTION BOUNDARY SCANNER 
# ==========================================
def find_adaptive_motion_boundaries(speeds, wrist_dists, elbow_angles, peak_pos, base_threshold=0.008):
    peak_speed = speeds[peak_pos]
    adaptive_threshold = max(0.003, min(base_threshold, peak_speed * 0.15))

    start = peak_pos
    while start > 0:
        window = speeds[max(0, start - 3):start]
        if len(window) > 0 and max(window) < adaptive_threshold:
            break
        start -= 1

    end = peak_pos
    while end < len(speeds) - 1:
        window = speeds[end + 1:min(len(speeds), end + 4)]
        if len(window) > 0 and max(window) < adaptive_threshold:
            if end + 3 < len(wrist_dists):
                dist_delta = abs(wrist_dists[end + 3] - wrist_dists[end])
                angle_delta = abs(elbow_angles[end + 3] - elbow_angles[end])
                if dist_delta < 0.01 and angle_delta < 3.0:
                    break
            else:
                break
        end += 1

    is_slow_punch = peak_speed < 0.035
    extra_post_pad = 12 if is_slow_punch else 0
    return start, end, extra_post_pad


# ==========================================
# 4. HIERARCHICAL INFERENCE
#    M1 == cross_rearhook -> M3 decides (Cross / Rear Hook)
#    M1 == anything else  -> M2 decides; M2 == NoPunch -> revert to M1
# ==========================================
def predict_hierarchical(clip_features, model1, model2, model3, timer):
    x1 = prepare_model_input(clip_features, model1)
    with timer.section("model: M1 predict"):
        p1 = run_model(model1, x1)
    m1_label = M1_CLASSES[int(np.argmax(p1))]

    if m1_label in M1_TO_M3:
        x3 = prepare_model_input(clip_features, model3)
        with timer.section("model: M3 predict"):
            p3 = run_model(model3, x3)
        return M3_CLASSES[int(np.argmax(p3))], m1_label, "M3"

    x2 = prepare_model_input(clip_features, model2)
    with timer.section("model: M2 predict"):
        p2 = run_model(model2, x2)
    m2_label = M2_CLASSES[int(np.argmax(p2))]

    if m2_label == "NoPunch":
        return M1_FALLBACK.get(m1_label, m1_label), m1_label, "M1-fallback"
    return m2_label, m1_label, "M2"


# ==========================================
# 5. MAIN
# ==========================================
def main():
    timer = Timer()

    t_load = time.perf_counter()
    print("Loading models...")
    specs = (
        ("M1", MODEL1_PATH, M1_CLASSES, FEATURE_DIM_M1),
        ("M2", MODEL2_PATH, M2_CLASSES, FEATURE_DIM_M2),
        ("M3", MODEL3_PATH, M3_CLASSES, FEATURE_DIM_M3),
    )
    loaded = {}
    for name, path, classes, fdim in specs:
        print(f"  {name}: {path}")
        m, how = load_punch_model(path, len(classes), fdim)
        assert m.output_shape[-1] == len(classes), \
            f"{name}: model has {m.output_shape[-1]} outputs but {len(classes)} classes are mapped"
        print(f"      loaded via {how} | input (T={m.input_shape[1]}, F={m.input_shape[-1]}), "
              f"outputs {m.output_shape[-1]}")
        run_model(m, np.zeros((1, m.input_shape[1] or MODEL_TARGET_FRAMES, m.input_shape[-1]), np.float32))
        loaded[name] = m
    model1, model2, model3 = loaded["M1"], loaded["M2"], loaded["M3"]
    load_time = time.perf_counter() - t_load
    print(f"Models loaded and warmed up in {load_time:.1f}s.")

    cap = cv2.VideoCapture(INPUT_VIDEO)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5, model_complexity=1)

    frame_buffer = deque(maxlen=200)
    norm_buffer = deque(maxlen=200)
    ankle_buffer = deque(maxlen=200)
    dist_l_buffer = deque(maxlen=200)
    dist_r_buffer = deque(maxlen=200)
    lead_wrist_y_buffer = deque(maxlen=200)
    lw_speed_buffer = deque(maxlen=200)
    rw_speed_buffer = deque(maxlen=200)
    l_elbow_buffer = deque(maxlen=200)
    r_elbow_buffer = deque(maxlen=200)

    prev_norm = None
    cooldown, punch_counter = 0, 0
    frame_idx = 0

    print("Running video trimming and classification...")
    pc = time.perf_counter
    t_wall_start = pc()

    while cap.isOpened():
        t0 = pc()
        ret, frame = cap.read()
        timer.add("io: video read", pc() - t0)
        if not ret:
            break
        frame_idx += 1

        t0 = pc()
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        timer.add("io: color convert", pc() - t0)

        t0 = pc()
        results = pose.process(rgb_frame)
        timer.add("mediapipe: pose.process", pc() - t0)

        if results.pose_landmarks:
            t0 = pc()
            lm = results.pose_landmarks.landmark
            dl, dr = get_wrist_distances(lm)
            lead_wrist_y_raw = lm[15].y

            raw_coords = []
            for i in LANDMARK_INDICES:
                raw_coords.extend([lm[i].x, lm[i].y])

            norm = scale_normalization(translation_normalization(raw_coords))
            lw_speed, rw_speed, l_elbow, r_elbow = compute_streaming_metrics(norm, prev_norm)
            prev_norm = norm
            timer.add("per-frame: normalize + metrics", pc() - t0)

            t0 = pc()
            frame_buffer.append(frame.copy())
            norm_buffer.append(norm)
            ankle_buffer.append((lm[27].x, lm[27].y, lm[28].x, lm[28].y))
            dist_l_buffer.append(dl)
            dist_r_buffer.append(dr)
            lead_wrist_y_buffer.append(lead_wrist_y_raw)
            lw_speed_buffer.append(lw_speed)
            rw_speed_buffer.append(rw_speed)
            l_elbow_buffer.append(l_elbow)
            r_elbow_buffer.append(r_elbow)
            timer.add("per-frame: buffer append", pc() - t0)

            if len(frame_buffer) >= TOTAL_WINDOW and cooldown == 0:
                t0 = pc()
                idx = -(WINDOW_AFTER + 1)
                vals_l = list(dist_l_buffer)
                vals_r = list(dist_r_buffer)
                vals_y = list(lead_wrist_y_buffer)
                vals_lw_speed = list(lw_speed_buffer)

                is_peak_l = vals_l[idx] >= max(vals_l[idx-5:idx+6]) * 0.97
                is_peak_r = vals_r[idx] >= max(vals_r[idx-5:idx+6]) * 0.97
                is_lead_peak = is_peak_l and vals_l[idx] > LEAD_THRESHOLD
                is_rear_peak = is_peak_r and vals_r[idx] > REAR_THRESHOLD

                nb_y = vals_y[idx-7:idx+8]
                val_y = vals_y[idx]
                is_y_minimum = len(nb_y) == 15 and val_y == min(nb_y)
                is_moving_up = len(nb_y) == 15 and val_y < vals_y[idx - 10]
                is_high_enough = val_y < UPPERCUT_Y_THRESHOLD
                has_major_rise = (len(vals_y) >= abs(idx - 20) and (vals_y[idx - 20] - val_y) >= 0.08)
                has_explosive_speed = vals_lw_speed[idx] > 0.04
                is_lead_uppercut = (is_y_minimum and is_moving_up and is_high_enough
                                    and has_major_rise and has_explosive_speed)
                timer.add("trigger detection", pc() - t0)

                if is_lead_peak or is_rear_peak or is_lead_uppercut:
                    punch_counter += 1

                    trigger = ("LEAD_HORIZONTAL" if is_lead_peak else
                               "REAR_HORIZONTAL" if is_rear_peak else
                               "LEAD_UPPERCUT_Y")

                    t0 = pc()
                    clip_frames_full = list(frame_buffer)[-TOTAL_WINDOW:]
                    clip_norm_full = list(norm_buffer)[-TOTAL_WINDOW:]
                    clip_ankle_full = list(ankle_buffer)[-TOTAL_WINDOW:]

                    speeds_in_window = (list(lw_speed_buffer)[-TOTAL_WINDOW:] if "LEAD" in trigger
                                        else list(rw_speed_buffer)[-TOTAL_WINDOW:])
                    wrist_dists_window = (list(dist_l_buffer)[-TOTAL_WINDOW:] if "LEAD" in trigger
                                          else list(dist_r_buffer)[-TOTAL_WINDOW:])
                    elbow_angles_window = (list(l_elbow_buffer)[-TOTAL_WINDOW:] if "LEAD" in trigger
                                           else list(r_elbow_buffer)[-TOTAL_WINDOW:])

                    peak_pos = WINDOW_BEFORE
                    motion_start, motion_end, extra_post_pad = find_adaptive_motion_boundaries(
                        speeds_in_window, wrist_dists_window, elbow_angles_window, peak_pos, VELOCITY_THRESHOLD
                    )

                    pre_pad = PRE_PAD_REAR if "REAR" in trigger else PRE_PAD_LEAD
                    post_pad = (POST_PAD_REAR if "REAR" in trigger else POST_PAD_LEAD) + extra_post_pad

                    crop_start = max(0, motion_start - pre_pad)
                    crop_end = min(TOTAL_WINDOW - 1, motion_end + post_pad)

                    clip_frames = clip_frames_full[crop_start:crop_end + 1]
                    clip_norm = clip_norm_full[crop_start:crop_end + 1]
                    clip_ankle = clip_ankle_full[crop_start:crop_end + 1]
                    timer.add("clip crop + boundaries", pc() - t0)

                    # ---- frame0-anchored 67-dim features for this clip ----
                    t0 = pc()
                    clip_feat_67 = process_punch_sequence(clip_norm)
                    timer.add("clip feature build", pc() - t0)

                    # ---- hierarchy ----
                    predicted_punch, m1_label, route = predict_hierarchical(
                        clip_feat_67, model1, model2, model3, timer
                    )

                    # ---- feedback (text only; no video overlay) ----
                    feedback_list = None
                    if ENABLE_FEEDBACK and predicted_punch != "NoPunch":
                        user_seq_55 = [np.asarray(v)[:55] for v in clip_feat_67]

                        t0 = pc()
                        refs = get_reference_data(predicted_punch)
                        timer.add("feedback: reference load", pc() - t0)

                        if refs is None:
                            feedback_list = [f"No reference .pkl data available for '{predicted_punch}'."]
                        elif len(user_seq_55) < 2:
                            feedback_list = ["Clip too short to analyze."]
                        else:
                            t0 = pc()
                            feedback_list, votes, n_refs, per_ref_logs = run_feedback_ensemble(
                                user_seq_55, refs,
                                LABEL_TO_PUNCH_TYPE.get(predicted_punch, predicted_punch.lower()),
                                clip_ankle
                            )
                            timer.add("feedback: ensemble DTW + metrics", pc() - t0)

                            t0 = pc()
                            needed = n_refs // 2 + 1
                            header = [f"Ensemble feedback: {n_refs} references, majority = {needed}", "Votes:"]
                            header += [f"  {c}/{n_refs}  {m}"
                                       for m, c in sorted(votes.items(), key=lambda kv: -kv[1])]
                            if not votes:
                                header.append("  (no reference fired any feedback)")
                            log_path = os.path.join(
                                OUTPUT_DIR, f"punch_{punch_counter}_{predicted_punch}_feedback.txt"
                            )
                            with open(log_path, "w", encoding="utf-8") as lf:
                                lf.write("\n".join(header) + "\n\n")
                                for src, log in per_ref_logs:
                                    lf.write(f"##### REFERENCE: {src} #####\n")
                                    lf.write("\n".join(log) + "\n\n")
                            timer.add("feedback: log write", pc() - t0)

                    # ---- export: plain cut clip, no drawing, no text ----
                    t0 = pc()
                    out_path = os.path.join(OUTPUT_DIR, f"punch_{punch_counter}_{predicted_punch}.mp4")
                    vw = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
                    for f in clip_frames:
                        vw.write(f)
                    vw.release()
                    timer.add("video export", pc() - t0)

                    print(f"[Punch {punch_counter}] M1: {m1_label} -> via {route} -> '{predicted_punch}' | {out_path}")
                    if feedback_list:
                        for fb in feedback_list:
                            print(f"    >> {fb}")
                    cooldown = WINDOW_AFTER + 20

        if cooldown > 0:
            cooldown -= 1

    cap.release()
    pose.close()
    wall_time = pc() - t_wall_start

    print(f"\nInference Complete. Extracted & Classified {punch_counter} punches.")
    print_timing_report(timer, wall_time, frame_idx, punch_counter, load_time,
                        os.path.join(OUTPUT_DIR, "timing_report.txt"))


if __name__ == "__main__":
    main()